# 04. Подготовка дневных датасетов для Yandex DataLens

Ноутбук берёт очищенные дневные данные из `data/processed/`, оставляет компактный набор столбцов для DataLens, отдельно сохраняет ОФЗ-ПД и ОФЗ-ПК, а затем присоединяет ключевую ставку Банка России.

Месячные датасеты в этом проекте не создаются и не сохраняются.


## 1. Импорт библиотек и пути проекта

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SUMMARY_DIR = DATA_DIR / "summary"
FIGURES_DIR = PROJECT_ROOT / "report" / "figures"

for path in [RAW_DIR, PROCESSED_DIR, SUMMARY_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = PROCESSED_DIR
REPORTS_DIR = SUMMARY_DIR

FIXED_DATALENS_DIR = OUTPUT_DIR / "fixed_datalens"
FLOATING_DATALENS_DIR = OUTPUT_DIR / "floating_datalens"
PER_BOND_DATALENS_DAILY_DIR = OUTPUT_DIR / "by_bond_datalens_daily"

for path in [FIXED_DATALENS_DIR, FLOATING_DATALENS_DIR, PER_BOND_DATALENS_DAILY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data dir:", PROCESSED_DIR.resolve())

Project root: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only
Processed data dir: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/processed


## 2. Загрузка очищенных дневных данных и ключевой ставки

In [3]:
DAILY_PATH = PROCESSED_DIR / "ofz_clean_daily_2016_2026.csv"
KEY_RATE_PATH = PROCESSED_DIR / "key_rate_2016_2026.csv"

for path in [DAILY_PATH, KEY_RATE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

df_daily = pd.read_csv(DAILY_PATH)
key_rate = pd.read_csv(KEY_RATE_PATH)

df_daily["TRADEDATE"] = pd.to_datetime(df_daily["TRADEDATE"], errors="coerce")
key_rate["date"] = pd.to_datetime(key_rate["date"], errors="coerce")
key_rate["key_rate"] = pd.to_numeric(
    key_rate["key_rate"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

key_rate = (
    key_rate[["date", "key_rate"]]
    .dropna(subset=["date", "key_rate"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("Daily clean shape:", df_daily.shape)
print("Key rate shape:", key_rate.shape)
display(df_daily.head())
display(key_rate.head())

Daily clean shape: (15522, 45)
Key rate shape: (2514, 2)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,96.0501,96.0501,95.2501,95.5993,95.5773,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,False,False,False,False,False,False,False,95.5993,95.5773,95.5773,8.99,8.98,8.99
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.6000,95.8000,95.3550,95.6201,95.6335,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,False,False,False,False,False,False,False,95.6201,95.6335,95.6335,8.98,8.98,8.98
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8888,95.9000,95.7000,95.9000,95.7770,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,False,False,False,False,False,False,False,95.9000,95.7770,95.7770,8.95,8.94,8.95
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8000,95.9500,95.2900,95.7000,95.7496,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,False,False,False,False,False,False,False,95.7000,95.7496,95.7496,8.96,8.97,8.96
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.9499,96.5999,95.7600,96.4500,96.3739,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,False,False,False,False,False,False,False,96.4500,96.3739,96.3739,8.86,8.85,8.86


,date,key_rate
0,2016-05-04 00:00:00+03:00,11.0
1,2016-05-05 00:00:00+03:00,11.0
2,2016-05-06 00:00:00+03:00,11.0
3,2016-05-10 00:00:00+03:00,11.0
4,2016-05-11 00:00:00+03:00,11.0


## 3. Расчёт показателя для ОФЗ-ПК

In [4]:
FLOATING_RATE_HARD_CAP = 60.0


def require_columns(df: pd.DataFrame, columns: list[str], dataset_name: str) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"В датасете {dataset_name} нет нужных колонок: {missing}")


def make_safe_issue_filename(issue_name: str, secid: str, suffix: str) -> str:
    safe_issue_name = str(issue_name).replace(" ", "_").replace("/", "_").replace("\\", "_")
    return f"{safe_issue_name}_{secid}_{suffix}.csv"


def add_floating_coupon_rate_estimate(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Добавляет оценочную текущую купонную ставку для ОФЗ-ПК.
    Оценка строится по изменению накопленного купонного дохода между соседними торговыми днями.
    '''
    result = df.copy()
    require_columns(
        result,
        ["TRADEDATE", "SECID", "COUPON_TYPE", "ACCINT", "FACEVALUE", "YIELD_FOR_ANALYSIS"],
        "daily_for_rate_estimation",
    )

    result["TRADEDATE"] = pd.to_datetime(result["TRADEDATE"], errors="coerce")
    result = result.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

    is_floating = result["COUPON_TYPE"].eq("floating")
    days_diff = result.groupby("SECID")["TRADEDATE"].diff().dt.days
    accint_diff = result.groupby("SECID")["ACCINT"].diff()

    raw_rate = accint_diff / days_diff / result["FACEVALUE"] * 365 * 100

    valid_rate = (
        is_floating
        & days_diff.gt(0)
        & accint_diff.gt(0)
        & result["FACEVALUE"].gt(0)
        & raw_rate.replace([np.inf, -np.inf], np.nan).notna()
        & raw_rate.gt(0)
        & raw_rate.le(FLOATING_RATE_HARD_CAP)
    )

    result["FLOATING_COUPON_RATE_EST_RAW"] = np.where(valid_rate, raw_rate, np.nan)

    def clean_single_floating_rate_series(s: pd.Series) -> pd.Series:
        clean = s.copy()
        values = clean.dropna()
        if len(values) >= 30:
            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1
            if pd.notna(iqr) and iqr > 0:
                lower = max(0.0, q1 - 3.0 * iqr)
                upper = min(FLOATING_RATE_HARD_CAP, q3 + 3.0 * iqr)
                clean = clean.where(clean.between(lower, upper))
        return clean.ffill().bfill()

    result["FLOATING_COUPON_RATE_EST"] = (
        result
        .groupby("SECID", group_keys=False)["FLOATING_COUPON_RATE_EST_RAW"]
        .transform(clean_single_floating_rate_series)
    )

    result.loc[~is_floating, "FLOATING_COUPON_RATE_EST"] = np.nan
    result.loc[~is_floating, "FLOATING_COUPON_RATE_EST_RAW"] = np.nan

    result["RATE_FOR_ANALYSIS"] = np.where(
        is_floating,
        result["FLOATING_COUPON_RATE_EST"],
        result["YIELD_FOR_ANALYSIS"],
    )

    return result


df_daily_rates = add_floating_coupon_rate_estimate(df_daily)

fixed_check = df_daily_rates["COUPON_TYPE"].eq("fixed")
floating_check = df_daily_rates["COUPON_TYPE"].eq("floating")

max_diff_fixed = (
    df_daily_rates.loc[fixed_check, "RATE_FOR_ANALYSIS"]
    - df_daily_rates.loc[fixed_check, "YIELD_FOR_ANALYSIS"]
).abs().max()

floating_missing_rate = int(df_daily_rates.loc[floating_check, "FLOATING_COUPON_RATE_EST"].isna().sum())
floating_zero_rate = int(df_daily_rates.loc[floating_check, "FLOATING_COUPON_RATE_EST"].eq(0).sum())

print("Max |RATE_FOR_ANALYSIS - YIELD_FOR_ANALYSIS| for fixed bonds:", max_diff_fixed)
print("Floating missing estimated rates:", floating_missing_rate)
print("Floating zero estimated rates:", floating_zero_rate)
display(df_daily_rates.head())

Max |RATE_FOR_ANALYSIS - YIELD_FOR_ANALYSIS| for fixed bonds: 0.0
Floating missing estimated rates: 0
Floating zero estimated rates: 0


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW,FLOATING_COUPON_RATE_EST_RAW,FLOATING_COUPON_RATE_EST,RATE_FOR_ANALYSIS
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,96.0501,96.0501,95.2501,95.5993,95.5773,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,False,False,False,False,False,False,False,95.5993,95.5773,95.5773,8.99,8.98,8.99,NaN,NaN,8.99
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.6000,95.8000,95.3550,95.6201,95.6335,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,False,False,False,False,False,False,False,95.6201,95.6335,95.6335,8.98,8.98,8.98,NaN,NaN,8.98
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8888,95.9000,95.7000,95.9000,95.7770,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,False,False,False,False,False,False,False,95.9000,95.7770,95.7770,8.95,8.94,8.95,NaN,NaN,8.95
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8000,95.9500,95.2900,95.7000,95.7496,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,False,False,False,False,False,False,False,95.7000,95.7496,95.7496,8.96,8.97,8.96,NaN,NaN,8.96
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.9499,96.5999,95.7600,96.4500,96.3739,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,False,False,False,False,False,False,False,96.4500,96.3739,96.3739,8.86,8.85,8.86,NaN,NaN,8.86


## 4. Формирование компактных дневных CSV для DataLens

In [5]:
COMBINED_DATALENS_COLUMNS = [
    "TRADEDATE",              # дата торгов
    "SECID",                  # уникальный код выпуска ОФЗ
    "SHORTNAME",              # короткое название выпуска
    "COUPON_TYPE",            # тип купона: fixed / floating
    "PLOT_GROUP",             # группа для графиков: short / medium / long / floating_coupon
    "PRICE_FOR_ANALYSIS",     # основная цена для анализа
    "YIELD_FOR_ANALYSIS",     # доходность к погашению из исходных данных
    "RATE_FOR_ANALYSIS",      # YTM для fixed, оценочная купонная ставка для floating
    "YEARS_TO_MATURITY",      # лет до погашения
    "VALUE",                  # денежный объём торгов, руб.
]

FIXED_DATALENS_COLUMNS = [
    "TRADEDATE",
    "SECID",
    "SHORTNAME",
    "COUPON_TYPE",
    "PLOT_GROUP",
    "PRICE_FOR_ANALYSIS",
    "YIELD_FOR_ANALYSIS",
    "RATE_FOR_ANALYSIS",
    "YEARS_TO_MATURITY",
    "DURATION",
    "VALUE",
]

FLOATING_DATALENS_COLUMNS = [
    "TRADEDATE",
    "SECID",
    "SHORTNAME",
    "COUPON_TYPE",
    "PLOT_GROUP",
    "PRICE_FOR_ANALYSIS",
    "ACCINT",
    "FLOATING_COUPON_RATE_EST",
    "RATE_FOR_ANALYSIS",
    "YEARS_TO_MATURITY",
    "VALUE",
]


def make_datalens_dataset(df: pd.DataFrame, columns: list[str], dataset_name: str) -> pd.DataFrame:
    require_columns(df, columns, dataset_name)
    result = df[columns].copy()
    result["TRADEDATE"] = pd.to_datetime(result["TRADEDATE"], errors="coerce")
    result = result.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)
    return result


df_datalens_daily = make_datalens_dataset(df_daily_rates, COMBINED_DATALENS_COLUMNS, "df_datalens_daily")
df_fixed_datalens_daily = make_datalens_dataset(
    df_daily_rates[df_daily_rates["COUPON_TYPE"].eq("fixed")],
    FIXED_DATALENS_COLUMNS,
    "df_fixed_datalens_daily",
)
df_floating_datalens_daily = make_datalens_dataset(
    df_daily_rates[df_daily_rates["COUPON_TYPE"].eq("floating")],
    FLOATING_DATALENS_COLUMNS,
    "df_floating_datalens_daily",
)

print("Combined daily DataLens shape:", df_datalens_daily.shape)
print("Fixed daily DataLens shape:", df_fixed_datalens_daily.shape)
print("Floating daily DataLens shape:", df_floating_datalens_daily.shape)
display(df_datalens_daily.head())

Combined daily DataLens shape: (15522, 10)
Fixed daily DataLens shape: (13751, 11)
Floating daily DataLens shape: (1771, 11)


,TRADEDATE,SECID,SHORTNAME,COUPON_TYPE,PLOT_GROUP,PRICE_FOR_ANALYSIS,YIELD_FOR_ANALYSIS,RATE_FOR_ANALYSIS,YEARS_TO_MATURITY,VALUE
0,2016-05-04,SU26207RMFS9,ОФЗ 26207,fixed,long,95.5773,8.99,8.99,10.751540,7.549269e+08
1,2016-05-05,SU26207RMFS9,ОФЗ 26207,fixed,long,95.6335,8.98,8.98,10.748802,1.133353e+08
2,2016-05-06,SU26207RMFS9,ОФЗ 26207,fixed,long,95.7770,8.95,8.95,10.746064,2.091805e+08
3,2016-05-10,SU26207RMFS9,ОФЗ 26207,fixed,long,95.7496,8.96,8.96,10.735113,2.660816e+07
4,2016-05-11,SU26207RMFS9,ОФЗ 26207,fixed,long,96.3739,8.86,8.86,10.732375,5.351472e+08


## 5. Присоединение ключевой ставки

In [8]:
OFZ_DATE_COL = "TRADEDATE"
KEY_RATE_COL = "key_rate"


def normalize_date_without_timezone(values: pd.Series) -> pd.Series:
    """
    Приводит даты к обычному datetime64[ns] без часового пояса.

    Важно: мы не переводим даты в UTC, а просто убираем timezone.
    Иначе дата вида 2016-05-03 00:00:00+03:00 может сдвинуться на 2016-05-02.
    """
    as_text = values.astype("string").str.strip()
    as_text = as_text.str.replace(r"([+-]\d{2}:?\d{2}|Z)$", "", regex=True)
    return pd.to_datetime(as_text, errors="coerce").dt.normalize()


key_rate_for_merge = (
    key_rate
    .rename(columns={"date": OFZ_DATE_COL})
    [[OFZ_DATE_COL, KEY_RATE_COL]]
    .copy()
)

key_rate_for_merge[OFZ_DATE_COL] = normalize_date_without_timezone(key_rate_for_merge[OFZ_DATE_COL])
key_rate_for_merge[KEY_RATE_COL] = pd.to_numeric(key_rate_for_merge[KEY_RATE_COL], errors="coerce")

key_rate_for_merge = (
    key_rate_for_merge
    .dropna(subset=[OFZ_DATE_COL, KEY_RATE_COL])
    .drop_duplicates(subset=[OFZ_DATE_COL])
    .sort_values(OFZ_DATE_COL)
    .reset_index(drop=True)
)


def add_key_rate_to_ofz(ofz_df: pd.DataFrame, key_rate_df: pd.DataFrame) -> pd.DataFrame:
    df = ofz_df.copy()

    df[OFZ_DATE_COL] = normalize_date_without_timezone(df[OFZ_DATE_COL])
    df = (
        df
        .dropna(subset=[OFZ_DATE_COL])
        .sort_values(OFZ_DATE_COL)
        .reset_index(drop=True)
    )

    rate_df = key_rate_df[[OFZ_DATE_COL, KEY_RATE_COL]].copy()
    rate_df[OFZ_DATE_COL] = normalize_date_without_timezone(rate_df[OFZ_DATE_COL])
    rate_df[KEY_RATE_COL] = pd.to_numeric(rate_df[KEY_RATE_COL], errors="coerce")

    rate_df = (
        rate_df
        .dropna(subset=[OFZ_DATE_COL, KEY_RATE_COL])
        .drop_duplicates(subset=[OFZ_DATE_COL])
        .sort_values(OFZ_DATE_COL)
        .reset_index(drop=True)
    )

    result = pd.merge_asof(
        df,
        rate_df,
        on=OFZ_DATE_COL,
        direction="backward",
    )

    if "YIELD_FOR_ANALYSIS" in result.columns:
        result["YIELD_KEY_RATE_SPREAD"] = result["YIELD_FOR_ANALYSIS"] - result[KEY_RATE_COL]

    if "RATE_FOR_ANALYSIS" in result.columns:
        result["RATE_KEY_RATE_SPREAD"] = result["RATE_FOR_ANALYSIS"] - result[KEY_RATE_COL]

    if "FLOATING_COUPON_RATE_EST" in result.columns:
        result["FLOATING_COUPON_KEY_RATE_SPREAD"] = result["FLOATING_COUPON_RATE_EST"] - result[KEY_RATE_COL]

    return result.sort_values(["SECID", OFZ_DATE_COL]).reset_index(drop=True)


df_datalens_daily_with_rate = add_key_rate_to_ofz(df_datalens_daily, key_rate_for_merge)
df_fixed_with_rate = add_key_rate_to_ofz(df_fixed_datalens_daily, key_rate_for_merge)
df_floating_with_rate = add_key_rate_to_ofz(df_floating_datalens_daily, key_rate_for_merge)

print("Combined daily with key rate:", df_datalens_daily_with_rate.shape)
print("Fixed daily with key rate:", df_fixed_with_rate.shape)
print("Floating daily with key rate:", df_floating_with_rate.shape)
print("Missing key_rate in combined:", int(df_datalens_daily_with_rate[KEY_RATE_COL].isna().sum()))

display(df_datalens_daily_with_rate.head())

Combined daily with key rate: (15522, 13)
Fixed daily with key rate: (13751, 14)
Floating daily with key rate: (1771, 14)
Missing key_rate in combined: 0


,TRADEDATE,SECID,SHORTNAME,COUPON_TYPE,PLOT_GROUP,PRICE_FOR_ANALYSIS,YIELD_FOR_ANALYSIS,RATE_FOR_ANALYSIS,YEARS_TO_MATURITY,VALUE,key_rate,YIELD_KEY_RATE_SPREAD,RATE_KEY_RATE_SPREAD
0,2016-05-04,SU26207RMFS9,ОФЗ 26207,fixed,long,95.5773,8.99,8.99,10.751540,7.549269e+08,11.0,-2.01,-2.01
1,2016-05-05,SU26207RMFS9,ОФЗ 26207,fixed,long,95.6335,8.98,8.98,10.748802,1.133353e+08,11.0,-2.02,-2.02
2,2016-05-06,SU26207RMFS9,ОФЗ 26207,fixed,long,95.7770,8.95,8.95,10.746064,2.091805e+08,11.0,-2.05,-2.05
3,2016-05-10,SU26207RMFS9,ОФЗ 26207,fixed,long,95.7496,8.96,8.96,10.735113,2.660816e+07,11.0,-2.04,-2.04
4,2016-05-11,SU26207RMFS9,ОФЗ 26207,fixed,long,96.3739,8.86,8.86,10.732375,5.351472e+08,11.0,-2.14,-2.14


## 6. Сохранение финальных дневных файлов

In [9]:
path_datalens_daily = OUTPUT_DIR / "ofz_for_datalens_daily_2016_2026.csv"
path_fixed_datalens_daily = FIXED_DATALENS_DIR / "ofz_fixed_for_datalens_daily_2016_2026.csv"
path_floating_datalens_daily = FLOATING_DATALENS_DIR / "ofz_floating_for_datalens_daily_2016_2026.csv"

path_fixed_with_rate = FIXED_DATALENS_DIR / "ofz_fixed_with_key_rate_for_datalens.csv"
path_floating_with_rate = FLOATING_DATALENS_DIR / "ofz_floating_with_key_rate_for_datalens.csv"

# В combined-файле сохраняется уже присоединённая ключевая ставка.
df_datalens_daily_with_rate.to_csv(path_datalens_daily, index=False, encoding="utf-8-sig")
df_fixed_datalens_daily.to_csv(path_fixed_datalens_daily, index=False, encoding="utf-8-sig")
df_floating_datalens_daily.to_csv(path_floating_datalens_daily, index=False, encoding="utf-8-sig")
df_fixed_with_rate.to_csv(path_fixed_with_rate, index=False, encoding="utf-8-sig")
df_floating_with_rate.to_csv(path_floating_with_rate, index=False, encoding="utf-8-sig")

saved_datalens_files = [
    {"file_type": "daily_all_bonds_with_key_rate", "rows": len(df_datalens_daily_with_rate), "columns": df_datalens_daily_with_rate.shape[1], "path": str(path_datalens_daily)},
    {"file_type": "daily_fixed_only", "rows": len(df_fixed_datalens_daily), "columns": df_fixed_datalens_daily.shape[1], "path": str(path_fixed_datalens_daily)},
    {"file_type": "daily_floating_only", "rows": len(df_floating_datalens_daily), "columns": df_floating_datalens_daily.shape[1], "path": str(path_floating_datalens_daily)},
    {"file_type": "daily_fixed_with_key_rate", "rows": len(df_fixed_with_rate), "columns": df_fixed_with_rate.shape[1], "path": str(path_fixed_with_rate)},
    {"file_type": "daily_floating_with_key_rate", "rows": len(df_floating_with_rate), "columns": df_floating_with_rate.shape[1], "path": str(path_floating_with_rate)},
]

for secid, part in df_datalens_daily_with_rate.groupby("SECID", sort=True):
    issue_name = part["SHORTNAME"].dropna().iloc[0] if part["SHORTNAME"].notna().any() else secid
    file_path = PER_BOND_DATALENS_DAILY_DIR / make_safe_issue_filename(issue_name, secid, "datalens_daily_2016_2026")
    part.to_csv(file_path, index=False, encoding="utf-8-sig")
    saved_datalens_files.append({
        "file_type": "daily_single_bond_with_key_rate",
        "rows": len(part),
        "columns": part.shape[1],
        "path": str(file_path),
    })

saved_datalens_files_report = pd.DataFrame(saved_datalens_files)
report_path = REPORTS_DIR / "datalens_daily_files_report_2016_2026.csv"
saved_datalens_files_report.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Saved compact daily DataLens files:")
display(saved_datalens_files_report)
print("Report:", report_path.resolve())

Saved compact daily DataLens files:


,file_type,rows,columns,path
0,daily_all_bonds_with_key_rate,15522,13,/Users/daniilsestov/Desktop/курсач для гита/of...
1,daily_fixed_only,13751,11,/Users/daniilsestov/Desktop/курсач для гита/of...
2,daily_floating_only,1771,11,/Users/daniilsestov/Desktop/курсач для гита/of...
3,daily_fixed_with_key_rate,13751,14,/Users/daniilsestov/Desktop/курсач для гита/of...
4,daily_floating_with_key_rate,1771,14,/Users/daniilsestov/Desktop/курсач для гита/of...
5,daily_single_bond_with_key_rate,2516,13,/Users/daniilsestov/Desktop/курсач для гита/of...
6,daily_single_bond_with_key_rate,2472,13,/Users/daniilsestov/Desktop/курсач для гита/of...
7,daily_single_bond_with_key_rate,1817,13,/Users/daniilsestov/Desktop/курсач для гита/of...
8,daily_single_bond_with_key_rate,1602,13,/Users/daniilsestov/Desktop/курсач для гита/of...
9,daily_single_bond_with_key_rate,1364,13,/Users/daniilsestov/Desktop/курсач для гита/of...


Report: /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/datalens_daily_files_report_2016_2026.csv
